# Proyecto de titulación — Preparación de datos para diagnóstico de fallas

**Autor del proyecto:** Jimmy Xavier Quinga Pillajo  
**Caso de estudio:** JAVERY, Ingeniería Humana y Tecnológica

Este notebook implementa un flujo reproducible de preparación de datos con **dos entornos aislados**:

1. **Entorno A — JAVERY:** datos reales de órdenes de servicio. Se preservan como evidencia institucional y se utilizan para limpieza, análisis descriptivo y validación operativa.
2. **Entorno B — Open Repair / CompCat:** datos públicos de computadoras con tipo de falla etiquetado. Se utilizan como corpus externo para el desarrollo experimental del modelo.

> **Regla metodológica:** los datos de JAVERY no se mezclan con el corpus público durante el entrenamiento inicial. La homologación se realiza únicamente sobre variables compatibles y conservando siempre la columna `fuente`.

**Variable objetivo homologada:** `tipo_falla`  
**Predictores comunes iniciales:** `tipo_equipo`, `marca`, `antiguedad_anios`, `sintoma`

Las variables posteriores al diagnóstico (componente dañado, solución, tiempo de reparación, costo y estado final) **no se usan como predictores**, para evitar *data leakage*.


## 1. Preparación del entorno

El notebook usa `pandas` para la limpieza, `matplotlib` para gráficos sencillos y `openpyxl` únicamente como motor de lectura de Excel dentro de Colab.


In [ ]:
!pip -q install openpyxl

import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

print("Entorno listo.")


# ENTORNO A — DATOS REALES JAVERY

## 2. Cargar el archivo original

Sube el archivo Excel original recibido de JAVERY. No lo edites antes de cargarlo. El notebook conserva una copia en memoria denominada `javery_original`.


In [ ]:
from google.colab import files

uploaded = files.upload()
xlsx_files = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]

if not xlsx_files:
    raise ValueError("No se cargó ningún archivo Excel.")

archivo_javery = xlsx_files[0]
javery_original = pd.read_excel(io.BytesIO(uploaded[archivo_javery]))

print("Archivo cargado:", archivo_javery)
print("Dimensiones originales:", javery_original.shape)
display(javery_original.head(10))


## 3. Normalizar nombres de columnas y excluir filas placeholder

En el archivo actual existen órdenes numeradas sin información adicional. Se conservan en el archivo original, pero no deben ingresar al dataset analítico.


In [ ]:
def normalizar_nombre_columna(c):
    c = str(c).strip().lower()
    c = (c.replace("á","a").replace("é","e").replace("í","i")
           .replace("ó","o").replace("ú","u").replace("ñ","n"))
    c = re.sub(r"\s+", "_", c)
    return c

javery = javery_original.copy()
javery.columns = [normalizar_nombre_columna(c) for c in javery.columns]

renombres = {
    "orden_de_servicio": "orden_servicio",
    "fechas": "fecha_ingreso",
    "tipo_de_equipo": "tipo_equipo_original",
    "marca": "marca_original",
    "modelo_antiguedad": "anio_equipo",
    "sintoma": "sintoma_original",
    "tipo_de_falla": "tipo_falla_original",
    "componente_danado": "componente_danado_original",
    "solucion": "solucion_original",
    "timepo_de_diagnostico": "tiempo_diagnostico_original",
    "tiempo_de_reparacion": "tiempo_reparacion_original",
    "costo": "costo_usd",
    "estado": "estado_original",
}
javery = javery.rename(columns=renombres)

campos_sustantivos = [
    "fecha_ingreso","tipo_equipo_original","marca_original","anio_equipo",
    "sintoma_original","tipo_falla_original"
]
existentes = [c for c in campos_sustantivos if c in javery.columns]

javery["fila_placeholder"] = javery[existentes].isna().all(axis=1)
javery = javery.loc[~javery["fila_placeholder"]].copy()

print("Registros útiles después de excluir placeholders:", len(javery))
display(javery)


## 4. Limpieza reproducible de campos

Se preservan los textos originales y se crean columnas limpias. No se reemplaza la evidencia primaria.


In [ ]:
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

# Fechas y año del equipo
javery["fecha_ingreso"] = pd.to_datetime(javery["fecha_ingreso"], errors="coerce")
javery["anio_equipo"] = pd.to_numeric(javery["anio_equipo"], errors="coerce")

# El campo 'modelo antigüedad' del archivo recibido contiene años (2010, 2018, 2020...).
javery["antiguedad_anios"] = javery["fecha_ingreso"].dt.year - javery["anio_equipo"]

# Tipo de equipo
map_equipo = {
    "pc": "desktop",
    "computador": "desktop",
    "computadora": "desktop",
    "desktop": "desktop",
    "laptop": "laptop",
    "portatil": "laptop",
    "portátil": "laptop",
    "celular": "celular",
    "impresora": "impresora",
}
javery["tipo_equipo_norm"] = (
    javery["tipo_equipo_original"].map(limpiar_texto).str.lower().map(map_equipo)
)

# Marca
javery["marca_norm"] = javery["marca_original"].map(limpiar_texto)
javery["marca_norm"] = javery["marca_norm"].replace({
    "s/m": "Sin marca", "S/M": "Sin marca",
    "hp": "HP", "lenovo": "Lenovo", "samsung": "Samsung", "epson": "Epson"
})

# Síntoma: limpieza mínima, sin cambiar significado
javery["sintoma_limpio"] = javery["sintoma_original"].map(limpiar_texto)
javery["sintoma_limpio"] = javery["sintoma_limpio"].replace({
    "impresión en regular estado": "impresión en regular estado",
    "no arranca el sistema operativo logo pegado": "no arranca el sistema operativo; logo detenido"
})

# Texto post-diagnóstico: solo para análisis, no como predictor
javery["componente_danado_limpio"] = javery["componente_danado_original"].map(limpiar_texto)
javery["solucion_limpia"] = javery["solucion_original"].map(limpiar_texto).replace({
    "cmbio de pantalla": "cambio de pantalla",
    "remplzao de las bobinas de poder": "reemplazo de las bobinas de poder",
    "remplzo del chip de video": "reemplazo del chip de video",
    "cargar todo el sistema operativo + programas basicos": "reinstalación del sistema operativo y programas básicos",
    "cargar todo el sistema operativo +  programas basicos": "reinstalación del sistema operativo y programas básicos",
})

# Tiempo de diagnóstico (todas las observaciones actuales están expresadas en horas)
javery["tiempo_diagnostico_horas"] = (
    javery["tiempo_diagnostico_original"].astype(str)
    .str.extract(r"(\d+(?:[\.,]\d+)?)", expand=False)
    .str.replace(",", ".", regex=False)
)
javery["tiempo_diagnostico_horas"] = pd.to_numeric(
    javery["tiempo_diagnostico_horas"], errors="coerce"
)

# Reparación: conservar unidad. No asumir que '1 día' equivale a 24 horas.
javery["tiempo_reparacion_std"] = (
    javery["tiempo_reparacion_original"].map(limpiar_texto)
    .str.replace(" dia", " día", regex=False)
    .str.replace(" dias", " días", regex=False)
)

javery["estado_norm"] = javery["estado_original"].map(limpiar_texto).str.capitalize()
javery["orden_servicio"] = javery["orden_servicio"].astype(str).str.strip().str.upper()
javery["fuente"] = "JAVERY"

display(javery)


## 5. Homologación de la variable objetivo a CompCat

La homologación se basa en la taxonomía oficial de fallas para computadoras de Open Repair / CompCat.

**Importante:** la homologación se aplica a partir del diagnóstico registrado, no del síntoma. Los registros nuevos cuya falla no esté mapeada se marcan como `REVISAR`, en lugar de asignar una clase automáticamente.


In [ ]:
# Mapeo inicial basado exclusivamente en las seis órdenes reales actualmente completas.
map_falla = {
    "pantalla rota": ("Integrated screen", 7),
    "cabezal tapado inyectores": ("Other", 13),
    "corto por sobre voltaje": ("System board", 18),
    "falla chip de video": ("System board", 18),
    "daño en la fuente de poder": ("Power/battery", 17),
    "eliminacion de archivos de arranque": ("Boot", 1),
    "eliminación de archivos de arranque": ("Boot", 1),
}

def homologar_falla(x):
    if pd.isna(x):
        return pd.Series([np.nan, np.nan, "REVISAR"])
    key = limpiar_texto(x).lower()
    if key in map_falla:
        nombre, fid = map_falla[key]
        return pd.Series([nombre, fid, "HOMOLOGADA"])
    return pd.Series([np.nan, np.nan, "REVISAR"])

javery[["tipo_falla_compcat","fault_type_id","estado_homologacion"]] = (
    javery["tipo_falla_original"].apply(homologar_falla)
)

# Compatibilidad de dominio con CompCat computers
javery["compatible_compcat"] = (
    javery["tipo_equipo_norm"].isin(["laptop","desktop"])
    & javery["tipo_falla_compcat"].notna()
)

javery["uso_recomendado"] = np.where(
    javery["compatible_compcat"],
    "Validación JAVERY",
    "Descriptivo / fuera de CompCat"
)

cols_revision = [
    "orden_servicio","tipo_equipo_norm","marca_norm","antiguedad_anios",
    "sintoma_limpio","tipo_falla_original","tipo_falla_compcat",
    "fault_type_id","compatible_compcat","uso_recomendado"
]
display(javery[cols_revision])


## 6. Control de calidad de JAVERY


In [ ]:
print("Registros reales útiles:", len(javery))
print("Compatibles con CompCat:", int(javery["compatible_compcat"].sum()))
print("Fuera del dominio CompCat:", int((~javery["compatible_compcat"]).sum()))
print("Pendientes de homologación:", int((javery["estado_homologacion"] == "REVISAR").sum()))

print("\nValores nulos por campo:")
display(javery.isna().sum().sort_values(ascending=False).to_frame("nulos"))

print("\nDistribución de equipos:")
display(javery["tipo_equipo_norm"].value_counts(dropna=False).to_frame("registros"))

print("\nDistribución de fallas homologadas:")
display(javery["tipo_falla_compcat"].value_counts(dropna=False).to_frame("registros"))


## 7. Exportar datasets JAVERY

Se generan dos archivos:

- `JAVERY_limpio.csv`: conserva toda la información útil y trazable.
- `JAVERY_validacion_orange.csv`: contiene únicamente las variables comunes y los registros compatibles con el modelo de computadoras.

Los registros de celular e impresora **no se eliminan de la base institucional**, pero no se usan para validar el modelo CompCat de laptops/desktops.


In [ ]:
columnas_limpias = [
    "orden_servicio","fecha_ingreso","tipo_equipo_original","tipo_equipo_norm",
    "marca_original","marca_norm","anio_equipo","antiguedad_anios",
    "sintoma_original","sintoma_limpio","tipo_falla_original",
    "tipo_falla_compcat","fault_type_id","componente_danado_limpio",
    "solucion_limpia","tiempo_diagnostico_horas","tiempo_reparacion_std",
    "costo_usd","estado_norm","fuente","compatible_compcat",
    "estado_homologacion","uso_recomendado"
]
javery_limpio = javery[[c for c in columnas_limpias if c in javery.columns]].copy()
javery_limpio.to_csv("JAVERY_limpio.csv", index=False, encoding="utf-8-sig")

javery_validacion = javery.loc[javery["compatible_compcat"], [
    "tipo_equipo_norm","marca_norm","antiguedad_anios",
    "sintoma_limpio","tipo_falla_compcat","fuente","orden_servicio"
]].copy()

javery_validacion.columns = [
    "tipo_equipo","marca","antiguedad_anios",
    "sintoma","tipo_falla","fuente","orden_servicio"
]
javery_validacion.to_csv("JAVERY_validacion_orange.csv", index=False, encoding="utf-8-sig")

print("Archivos generados:")
print("- JAVERY_limpio.csv:", javery_limpio.shape)
print("- JAVERY_validacion_orange.csv:", javery_validacion.shape)
display(javery_validacion)


# ENTORNO B — OPEN REPAIR / COMPCAT

## 8. Descargar las fuentes públicas oficiales

Fuentes utilizadas:

- `compcat_laptops.csv`
- `compcat_desktops.csv`
- `compcat_fault_types.csv`

La fuente externa se mantiene separada de JAVERY mediante la columna `fuente`.


In [ ]:
URL_LAPTOPS = "https://raw.githubusercontent.com/openrepair/data/master/quests/computers/compcat_laptops.csv"
URL_DESKTOPS = "https://raw.githubusercontent.com/openrepair/data/master/quests/computers/compcat_desktops.csv"
URL_FAULTS = "https://raw.githubusercontent.com/openrepair/data/master/quests/computers/compcat_fault_types.csv"

ora_laptop_raw = pd.read_csv(URL_LAPTOPS)
ora_desktop_raw = pd.read_csv(URL_DESKTOPS)
fault_catalog = pd.read_csv(URL_FAULTS)

print("Laptops:", ora_laptop_raw.shape)
print("Desktops:", ora_desktop_raw.shape)
print("Catálogo de fallas:", fault_catalog.shape)

display(fault_catalog)


## 9. Limpieza independiente de Open Repair

Criterios iniciales:

- Conservar solo registros con `problem` y `fault_type`.
- Excluir `Unknown` como objetivo de entrenamiento, porque no representa una falla determinada.
- Conservar el texto original y crear una copia limpia.
- Normalizar categoría de equipo y marca.
- Calcular antigüedad cuando existan año de fabricación y fecha del evento.
- No usar `repair_status` como predictor, porque puede reflejar información posterior al diagnóstico.


In [ ]:
ora = pd.concat([ora_laptop_raw, ora_desktop_raw], ignore_index=True)
ora_original = ora.copy()

ora["problem_original"] = ora["problem"]
ora["sintoma"] = ora["problem"].map(limpiar_texto)
ora["event_date"] = pd.to_datetime(ora["event_date"], errors="coerce")
ora["year_of_manufacture"] = pd.to_numeric(ora["year_of_manufacture"], errors="coerce")

ora["antiguedad_anios"] = ora["event_date"].dt.year - ora["year_of_manufacture"]

# Edades imposibles o muy atípicas se dejan como nulas para revisión.
ora.loc[(ora["antiguedad_anios"] < 0) | (ora["antiguedad_anios"] > 50), "antiguedad_anios"] = np.nan

ora["tipo_equipo"] = ora["product_category"].replace({
    "Laptop": "laptop",
    "Desktop computer": "desktop"
})

ora["marca"] = ora["brand"].map(limpiar_texto)
ora["marca"] = ora["marca"].replace({"Unknown": "Sin marca", "unknown": "Sin marca"})
ora["fuente"] = "OPEN_REPAIR_COMPCAT"

# Auditoría antes de filtrar
print("Duplicados completos:", ora.duplicated().sum())
print("Sin síntoma:", ora["sintoma"].isna().sum())
print("Sin tipo de falla:", ora["fault_type"].isna().sum())
print("Objetivo Unknown:", (ora["fault_type"] == "Unknown").sum())

ora_limpio = ora.drop_duplicates().copy()
ora_limpio = ora_limpio[ora_limpio["sintoma"].notna()].copy()
ora_limpio = ora_limpio[ora_limpio["fault_type"].notna()].copy()
ora_limpio = ora_limpio[ora_limpio["fault_type"] != "Unknown"].copy()

ora_limpio = ora_limpio.rename(columns={"fault_type": "tipo_falla"})

print("Dimensiones limpias:", ora_limpio.shape)
display(ora_limpio.head())


## 10. Auditoría de clases y dominio externo

No se eliminan clases minoritarias automáticamente. Primero se cuantifica el desbalance para decidir posteriormente el tratamiento en Orange.


In [ ]:
dist_fallas = ora_limpio["tipo_falla"].value_counts().sort_values(ascending=False)
display(dist_fallas.to_frame("registros"))

plt.figure(figsize=(10, 6))
dist_fallas.sort_values().plot(kind="barh")
plt.title("Distribución de tipos de falla — Open Repair / CompCat")
plt.xlabel("Registros")
plt.ylabel("Tipo de falla")
plt.tight_layout()
plt.show()

print("\nDistribución por tipo de equipo:")
display(ora_limpio["tipo_equipo"].value_counts().to_frame("registros"))

print("\nNulos en variables comunes:")
display(
    ora_limpio[["tipo_equipo","marca","antiguedad_anios","sintoma","tipo_falla"]]
    .isna().sum().to_frame("nulos")
)


## 11. Dataset externo para Orange

Se exporta con las mismas variables conceptuales de JAVERY. `id_ords` se conserva solo como identificador y no debe configurarse como predictor.


In [ ]:
openrepair_orange = ora_limpio[[
    "tipo_equipo","marca","antiguedad_anios","sintoma",
    "tipo_falla","fuente","id_ords"
]].copy()

openrepair_orange.to_csv(
    "OPEN_REPAIR_entrenamiento_orange.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset externo listo:", openrepair_orange.shape)
display(openrepair_orange.head())


# ENTORNO C — HOMOLOGACIÓN Y CONTROL EXPERIMENTAL

## 12. Verificar compatibilidad de esquemas

En esta etapa **no se concatena JAVERY al conjunto de entrenamiento**. Solo se comprueba que ambos archivos tengan las mismas variables necesarias para el experimento.


In [ ]:
vars_modelo = ["tipo_equipo","marca","antiguedad_anios","sintoma","tipo_falla"]

print("Columnas modelo Open Repair:")
print(openrepair_orange[vars_modelo].dtypes)

print("\nColumnas modelo JAVERY:")
print(javery_validacion[vars_modelo].dtypes)

print("\nClases JAVERY presentes en Open Repair:")
clases_ora = set(openrepair_orange["tipo_falla"].dropna().unique())
clases_jav = set(javery_validacion["tipo_falla"].dropna().unique())

print("JAVERY:", sorted(clases_jav))
print("Todas presentes en Open Repair:", clases_jav.issubset(clases_ora))
print("Faltantes:", sorted(clases_jav - clases_ora))


## 13. Regla anti-*data leakage*

Para el entrenamiento inicial, usar únicamente variables disponibles **antes o durante la recepción/diagnóstico preliminar**:

- `tipo_equipo`
- `marca`
- `antiguedad_anios`
- `sintoma`

Variable objetivo:

- `tipo_falla`

No utilizar como predictores:

- componente dañado
- diagnóstico técnico textual que revele la respuesta
- solución aplicada
- tiempo de diagnóstico
- tiempo de reparación
- costo
- estado final
- `repair_status`

Estas variables pueden mantenerse para análisis descriptivo y dashboards de Power BI.


## 14. Nota importante sobre el texto multilingüe

Open Repair contiene descripciones en varios idiomas, mientras que JAVERY registra síntomas en español. Antes de interpretar el rendimiento final, debe evaluarse este **cambio de dominio e idioma**.

Para Orange existen dos rutas posibles:

1. **Orange Text Mining:** vectorizar el campo `sintoma` y entrenar los clasificadores con las características textuales.
2. **Ingeniería de características externa:** generar representaciones numéricas multilingües en Colab y exportarlas a Orange.

No se debe asumir que el texto bruto multilingüe tendrá el mismo comportamiento que las descripciones en español de JAVERY.


## 15. Archivos finales y descarga


In [ ]:
archivos_salida = [
    "JAVERY_limpio.csv",
    "JAVERY_validacion_orange.csv",
    "OPEN_REPAIR_entrenamiento_orange.csv",
]

for nombre in archivos_salida:
    print(nombre)

# Descomenta si deseas descargar automáticamente:
# from google.colab import files
# for nombre in archivos_salida:
#     files.download(nombre)


# Resultado metodológico

Al finalizar este notebook se dispone de:

- una base institucional JAVERY limpia y trazable;
- un subconjunto JAVERY compatible reservado para validación;
- un corpus Open Repair/CompCat limpio para entrenamiento experimental;
- una taxonomía común de tipo de falla;
- una separación explícita entre datos reales y datos públicos;
- controles para evitar *data leakage*;
- archivos CSV con estructura compatible para el siguiente paso en Orange.

**No debe afirmarse que Open Repair forma parte de las órdenes históricas de JAVERY.** En la memoria técnica debe declararse como una fuente pública complementaria y metodológicamente independiente.
